In [2]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms
import torchinfo
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torch.utils.data import Dataset

import helper_utils


In [3]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: CUDA")
else:
    device = torch.device("cpu")
    print(f"Using device: CPU")

Using device: CUDA


## Hyperparameters

In [4]:
BATCH_SIZE = 32

## Datasets and DataLoaders

### Dataset Transforms

In [5]:
train_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5] * 3, std=[0.5] * 3)
])

val_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5] * 3, std=[0.5] * 3)
])

### Train, Test and Validation Datasets and DataLoaders

In [8]:
train_dir = os.path.join("data", "train")
test_dir = os.path.join("data", "test")
val_dir = os.path.join("data", "validation")

train_dataset = ImageFolder(root = train_dir, transform = None)
val_dataset = ImageFolder(root = val_dir, transform = None)
test_dataset = ImageFolder(root = test_dir, transform = None)

train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = BATCH_SIZE, shuffle = False)
test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE, shuffle = False)

In [9]:
classes = train_dataset.classes

num_classes = len(classes)

print(f"Classes: {classes}")
print(f"Number of classes: {num_classes}")

Classes: ['dress', 'hat', 'longsleeve', 'outwear', 'pants', 'shirt', 'shoes', 'shorts', 'skirt', 't-shirt']
Number of classes: 10


In [ ]:
class InvertedResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride, expansion_factor, shortcut = None):
        super(InvertedResidualBlock, self).__init__()

        hidden_dim = in_channels * expansion_factor



        # 1 x 1 pointwise convolution to expand the number of channels
        self.expand = nn.Sequential(
            nn.Conv2d(in_channels =  in_channels,
                      out_channels = hidden_dim,
                      kernel_size = 1, 
                      stride = stride,
                      padding = 0,
                      bias = False
                      ),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(inplace = True)
        )

        # 3 x 3 depthwise convolution
        self.depthwise = nn.Sequential(
            nn.Conv2d(in_channels = hidden_dim,
                      out_channels = hidden_dim,
                      kernel_size = 3,
                      stride = stride,
                      padding = 1,
                      bias = False,
                      groups = hidden_dim
                      ),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(inplace = True)
        )

        # 1 x 1 pointwise convolution to project back to the original number of channels
        self.project = nn.Sequential(
            nn.Conv2d(in_channels = hidden_dim,
                      out_channels = out_channels,
                      kernel_size = 1,
                      bias = False)
        )
